In [ ]:
!pip install transformers==4.57.6
!pip install simpletransformers==0.70.5

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import classification_report
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import matplotlib.pyplot as plt
import seaborn as sn

In [20]:
imdb_path = 'topic-datasets/IMDB Dataset.csv'
kindle_path = 'topic-datasets/preprocessed_kindle_review .csv'

# Load datasets
imdb_full = pd.read_csv(imdb_path)
kindle_full = pd.read_csv(kindle_path)
restaurant_full = pd.read_csv('topic-datasets/Restaurant_Reviews.tsv', sep='\t')
test_raw = pd.read_csv('topic-datasets/Sentiment-topic-test.tsv', sep='\t')
topic_mapping = { 'movie': 0, 'book': 1, 'restaurant': 2}

# Sample exactly 1000 random instances reproducibly for each class
imdb_sample = imdb_full.sample(n=1000, random_state=42)
kindle_sample = kindle_full.sample(n=1000, random_state=42)

#Keep relevant columns for topic classification task
imdb_clean = pd.DataFrame({'text': imdb_sample['review'],  'labels': 0 })
kindle_clean = pd.DataFrame({ 'text': kindle_sample['reviewText'], 'labels': 1 })
restaurant_clean = pd.DataFrame({ 'text': restaurant_full['Review'], 'labels': 2 })

all_training_data = pd.concat([imdb_clean, kindle_clean, restaurant_clean], ignore_index=True)
train = all_training_data.sample(frac=1, random_state=42).reset_index(drop=True)
train = train.dropna().reset_index(drop=True)

print(df.head())
print("\nClass distribution:")
print(df['labels'].value_counts())


                                                text  labels
0  This is a pretty good story,has a little of ev...       1
1  This is not a book to be set aside lightly. Th...       1
2  This is one of the most dissapointing purchase...       1
3  Alexander Nevsky (1938) is a brilliant piece o...       0
4                  I would not recommend this place.       2

Class distribution:
labels
1    1000
0    1000
2    1000
Name: count, dtype: int64


In [22]:
from sklearn.model_selection import train_test_split

test = pd.DataFrame({ 'text': test_raw['text'], 'labels': test_raw['topic'].map(topic_mapping)})
train, dev = train_test_split(train, test_size=0.1, random_state=42, stratify=train['labels'])

print(test.head())


                                                text  labels
0  It took eight years for Warner Brothers to rec...       0
1  All the New York University students love this...       2
2  This Italian place is really trendy but they h...       2
3  In conclusion, my review of this book would be...       1
4  The story of this movie is focused on Carl Bra...       0


In [ ]:
# Model configuration # https://simpletransformers.ai/docs/usage/#configuring-a-simple-transformers-model
model_args = ClassificationArgs()

model_args.overwrite_output_dir=True # overwrite existing saved models in the same directory
model_args.evaluate_during_training=True # to perform evaluation while training the model
# (eval data should be passed to the training method)

model_args.num_train_epochs=10 # number of epochs
model_args.train_batch_size=32 # batch size
model_args.learning_rate=4e-6 # learning rate
model_args.max_seq_length=256 # maximum sequence length
# Note! Increasing max_seq_len may provide better performance, but training time will increase.
# For educational purposes, we set max_seq_len to 256.

# Early stopping to combat overfitting: https://simpletransformers.ai/docs/tips-and-tricks/#using-early-stopping
model_args.use_early_stopping=True
model_args.early_stopping_delta=0.01 # "The improvement over best_eval_loss necessary to count as a better checkpoint"
model_args.early_stopping_metric='eval_loss'
model_args.early_stopping_metric_minimize=True
model_args.early_stopping_patience=2
model_args.evaluate_during_training_steps=32 # how often you want to run validation in terms of training steps (or batches)

# Checking steps per epoch
steps_per_epoch = int(np.ceil(len(train) / float(model_args.train_batch_size)))
print('Each epoch will have {:,} steps.'.format(steps_per_epoch)) # 64 steps = validating 2 times per epoch

In [ ]:
model = ClassificationModel('bert', 'bert-base-cased', num_labels=4, args=model_args, use_cuda=True) # CUDA is enabled

In [ ]:
print(str(model.args).replace(',', '\n')) # model args

In [ ]:
_, history = model.train_model(train, eval_df=dev)

In [ ]:
train_loss = history['train_loss']
eval_loss = history['eval_loss']
plt.plot(train_loss, label='Training loss')
plt.plot(eval_loss, label='Evaluation loss')
plt.title('Training and evaluation loss')
plt.legend()

In [ ]:
# Evaluate the model
result, model_outputs, wrong_predictions = model.eval_model(dev)
result

In [ ]:
predicted, probabilities = model.predict(test.text.to_list())
test['predicted'] = predicted

In [ ]:
print(classification_report(test['labels'], test['predicted']))